# Exercise 10: Dynamic Overwrite (MERGE-like) in Iceberg

Update partitions without truncating/deleting the whole table, using `overwriteDynamic` and `MERGE INTO`

In [0]:
# Step 0: DataFrame creation with updated data
from pyspark.sql import Row

updates = [
    Row(country="MX", year=2024, value=999),   # update
    Row(country="CA", year=2024, value=50),    # new row
]

df_updates = spark.createDataFrame(updates)
df_updates.show()

+-------+----+-----+
|country|year|value|
+-------+----+-----+
|     MX|2024|  999|
|     CA|2024|   50|
+-------+----+-----+



In [0]:
# Step 1: Dynamic Overwrite (It will just replace affected partitions)
(
    df_updates.write
      .format("iceberg")
      .mode("overwrite")
      .option("overwrite-mode", "dynamic")
      .saveAsTable("workspace.default.iceberg_partitioned")
)

In [0]:
%sql
-- Step 2: Validation
SELECT * FROM workspace.default.iceberg_partitioned ORDER BY country, year;

In [0]:
%sql
-- Step 1.1: MERGE INTO
MERGE INTO workspace.default.iceberg_partitioned AS t
USING (
  SELECT "US" AS country, 2025 AS year, 777 AS value
) AS s
ON t.country = s.country AND t.year = s.year
WHEN MATCHED THEN UPDATE SET value = s.value
WHEN NOT MATCHED THEN INSERT (country, year, value) VALUES (s.country, s.year, s.value);

In [0]:
%sql
-- Step 2.1: MERGE Validation
SELECT * FROM workspace.default.iceberg_partitioned WHERE country = 'US' AND year = 2025;

In [0]:
%sql
-- Step 3.1: Check the new Snapshot
CALL workspace.system.snapshots('workspace.default.iceberg_partitioned');